# SIC/XE Assembler 

Input parser, OPTAB, Pass 1, location counter, and SYMTAB.

In [ ]:

This module implements the source-line parser, OPTAB, Pass 1, the location
counter, and the symbol table. Pass 2 and the teammate features remain outside
this module.
"""

from dataclasses import dataclass, asdict
import ast
import re


# mnemonic: (hex opcode, instruction format)
# A format of 3 also allows extended format 4 when the source uses +MNEMONIC.
OPTAB = {
    "ADD": ("18", 3), "ADDF": ("58", 3), "ADDR": ("90", 2),
    "AND": ("40", 3), "CLEAR": ("B4", 2), "COMP": ("28", 3),
    "COMPF": ("88", 3), "COMPR": ("A0", 2), "DIV": ("24", 3),
    "DIVF": ("64", 3), "DIVR": ("9C", 2), "FIX": ("C4", 1),
    "FLOAT": ("C0", 1), "HIO": ("F4", 1), "J": ("3C", 3),
    "JEQ": ("30", 3), "JGT": ("34", 3), "JLT": ("38", 3),
    "JSUB": ("48", 3), "LDA": ("00", 3), "LDB": ("68", 3),
    "LDCH": ("50", 3), "LDF": ("70", 3), "LDL": ("08", 3),
    "LDS": ("6C", 3), "LDT": ("74", 3), "LDX": ("04", 3),
    "LPS": ("D0", 3), "MUL": ("20", 3), "MULF": ("60", 3),
    "MULR": ("98", 2), "NORM": ("C8", 1), "OR": ("44", 3),
    "RD": ("D8", 3), "RMO": ("AC", 2), "RSUB": ("4C", 3),
    "SHIFTL": ("A4", 2), "SHIFTR": ("A8", 2), "SIO": ("F0", 1),
    "SSK": ("EC", 3), "STA": ("0C", 3), "STB": ("78", 3),
    "STCH": ("54", 3), "STF": ("80", 3), "STI": ("D4", 3),
    "STL": ("14", 3), "STS": ("7C", 3), "STSW": ("E8", 3),
    "STT": ("84", 3), "STX": ("10", 3), "SUB": ("1C", 3),
    "SUBF": ("5C", 3), "SUBR": ("94", 2), "SVC": ("B0", 2),
    "TD": ("E0", 3), "TIO": ("F8", 1), "TIX": ("2C", 3),
    "TIXR": ("B8", 2), "WD": ("DC", 3),
}


DIRECTIVES = {
    "START", "END", "BYTE", "WORD", "RESB", "RESW", "BASE",
    "NOBASE", "EQU", "ORG", "LTORG", "USE", "CSECT", "EXTDEF",
    "EXTREF", "MACRO", "MEND",
}


@dataclass
class SourceLine:
    line_number: int
    raw: str
    label: str = ""
    opcode: str = ""
    operand: str = ""
    comment: str = ""
    address: int | None = None
    error: str = ""


def split_comment(line):
    """Split code from a period-style SIC/XE comment, respecting quotes."""
    in_quote = False
    for index, character in enumerate(line):
        if character == "'":
            in_quote = not in_quote
        elif character == "." and not in_quote:
            if index == 0 or line[index - 1].isspace():
                return line[:index].rstrip(), line[index:].strip()
    return line.rstrip(), ""


def parse_source_line(line, line_number):
    """Convert one source line into label, opcode, operand, and comment fields."""
    raw = line.rstrip("\n")
    code, comment = split_comment(raw)

    if not code.strip():
        return SourceLine(line_number, raw, comment=comment)

    fields = code.split()
    first = fields[0].upper()
    first_opcode = first[1:] if first.startswith("+") else first

    # If the first token is an operation/directive, the line has no label.
    if first_opcode in OPTAB or first in DIRECTIVES:
        label = ""
        opcode = fields[0]
        operand = " ".join(fields[1:])
    else:
        label = fields[0]
        opcode = fields[1] if len(fields) > 1 else ""
        operand = " ".join(fields[2:]) if len(fields) > 2 else ""

    return SourceLine(
        line_number=line_number,
        raw=raw,
        label=label.upper(),
        opcode=opcode.upper(),
        operand=operand,
        comment=comment,
    )


def byte_length(operand):
    """Return the number of bytes reserved by a BYTE directive."""
    match = re.fullmatch(r"([CXcx])'([^']*)'", operand.strip())
    if not match:
        raise ValueError("Invalid BYTE constant")

    constant_type, value = match.groups()
    if constant_type.upper() == "C":
        return len(value)

    if len(value) % 2 != 0 or not re.fullmatch(r"[0-9A-Fa-f]*", value):
        raise ValueError("Hex BYTE constants need an even number of hex digits")
    return len(value) // 2


def decimal_operand(operand, directive):
    try:
        value = int(operand.strip(), 10)
    except ValueError as error:
        raise ValueError(f"{directive} requires a decimal integer") from error
    if value < 0:
        raise ValueError(f"{directive} cannot reserve negative storage")
    return value


class ExpressionEvaluator(ast.NodeVisitor):
    """Safely evaluate a simple EQU expression using existing symbols."""

    def __init__(self, symbols, current_location):
        self.symbols = symbols
        self.current_location = current_location

    def evaluate(self, expression):
        expression = expression.strip()
        if expression == "*":
            return self.current_location
        tree = ast.parse(expression, mode="eval")
        return self.visit(tree.body)

    def visit_Name(self, node):
        name = node.id.upper()
        if name not in self.symbols:
            raise ValueError(f"Undefined symbol in EQU expression: {name}")
        return self.symbols[name]

    def visit_Constant(self, node):
        if isinstance(node.value, int):
            return node.value
        raise ValueError("EQU expressions may contain only integers and symbols")

    def visit_UnaryOp(self, node):
        value = self.visit(node.operand)
        if isinstance(node.op, ast.UAdd):
            return value
        if isinstance(node.op, ast.USub):
            return -value
        raise ValueError("Unsupported operator in EQU expression")

    def visit_BinOp(self, node):
        left = self.visit(node.left)
        right = self.visit(node.right)
        if isinstance(node.op, ast.Add):
            return left + right
        if isinstance(node.op, ast.Sub):
            return left - right
        if isinstance(node.op, ast.Mult):
            return left * right
        if isinstance(node.op, (ast.Div, ast.FloorDiv)):
            if right == 0:
                raise ValueError("Division by zero in EQU expression")
            return left // right
        raise ValueError("Unsupported operator in EQU expression")

    def generic_visit(self, node):
        raise ValueError("Unsupported value in EQU expression")


class AssemblerPass1:
    def __init__(self, optab=None):
        self.optab = dict(optab or OPTAB)
        self.symtab = {}
        self.intermediate = []
        self.errors = []
        self.program_name = "NONAME"
        self.start_address = 0
        self.locctr = 0
        self.program_length = 0
        self.end_operand = ""

    def add_error(self, source_line, message):
        source_line.error = message
        self.errors.append(f"Line {source_line.line_number}: {message}")

    def add_symbol(self, source_line, value=None):
        if not source_line.label:
            return
        if source_line.label in self.symtab:
            self.add_error(source_line, f"Duplicate symbol: {source_line.label}")
            return
        self.symtab[source_line.label] = self.locctr if value is None else value

    def instruction_length(self, source_line):
        opcode = source_line.opcode
        extended = opcode.startswith("+")
        mnemonic = opcode[1:] if extended else opcode

        if mnemonic not in self.optab:
            raise ValueError(f"Invalid operation code: {opcode}")

        standard_format = self.optab[mnemonic][1]
        if extended:
            if standard_format != 3:
                raise ValueError(f"{mnemonic} cannot use format 4")
            return 4
        return standard_format

    def process_line(self, source_line):
        if not source_line.opcode:
            self.intermediate.append(source_line)
            return False

        opcode = source_line.opcode
        source_line.address = self.locctr

        if opcode == "START":
            if source_line.operand:
                try:
                    self.start_address = int(source_line.operand, 16)
                    self.locctr = self.start_address
                    source_line.address = self.locctr
                except ValueError:
                    self.add_error(source_line, "START requires a hexadecimal address")
            self.program_name = source_line.label or "NONAME"
            self.intermediate.append(source_line)
            return False

        if opcode == "END":
            self.end_operand = source_line.operand.strip().upper()
            self.intermediate.append(source_line)
            return True

        if opcode == "EQU":
            if not source_line.label:
                self.add_error(source_line, "EQU requires a label")
            else:
                try:
                    value = ExpressionEvaluator(self.symtab, self.locctr).evaluate(
                        source_line.operand
                    )
                    self.add_symbol(source_line, value)
                except (ValueError, SyntaxError) as error:
                    self.add_error(source_line, str(error))
            self.intermediate.append(source_line)
            return False

        self.add_symbol(source_line)

        try:
            mnemonic = opcode[1:] if opcode.startswith("+") else opcode
            if mnemonic in self.optab:
                self.locctr += self.instruction_length(source_line)
            elif opcode == "WORD":
                self.locctr += 3
            elif opcode == "RESW":
                self.locctr += 3 * decimal_operand(source_line.operand, "RESW")
            elif opcode == "RESB":
                self.locctr += decimal_operand(source_line.operand, "RESB")
            elif opcode == "BYTE":
                self.locctr += byte_length(source_line.operand)
            elif opcode in {"BASE", "NOBASE", "EXTDEF", "EXTREF"}:
                pass
            elif opcode in {"LTORG", "USE", "CSECT", "MACRO", "MEND", "ORG"}:
                self.add_error(
                    source_line,
                    f"{opcode} belongs to a teammate/shared project module and is not handled here",
                )
            else:
                self.add_error(source_line, f"Invalid operation code: {opcode}")
        except ValueError as error:
            self.add_error(source_line, str(error))

        self.intermediate.append(source_line)
        return False

    def run(self, file_name="functions.txt"):
        with open(file_name, "r", encoding="utf-8") as source_file:
            for line_number, line in enumerate(source_file, start=1):
                source_line = parse_source_line(line, line_number)
                if self.process_line(source_line):
                    break

        self.program_length = self.locctr - self.start_address
        return self.results()

    def results(self):
        return {
            "program_name": self.program_name,
            "start_address": self.start_address,
            "program_length": self.program_length,
            "final_locctr": self.locctr,
            "symtab": dict(self.symtab),
            "intermediate": [asdict(line) for line in self.intermediate],
            "errors": list(self.errors),
        }


def print_pass1_results(results):
    print("PASS 1 RESULTS")
    print("Program name:", results["program_name"])
    print(f'Start address: {results["start_address"]:04X}')
    print(f'Program length: {results["program_length"]:04X}')
    print(f'Final LOCCTR: {results["final_locctr"]:04X}')

    print("\nSYMBOL TABLE")
    if results["symtab"]:
        for symbol, address in sorted(results["symtab"].items()):
            print(f"{symbol:<12} {address:04X}")
    else:
        print("No symbols were found.")

    print("\nERRORS")
    if results["errors"]:
        for error in results["errors"]:
            print(error)
    else:
        print("No Pass 1 errors found.")


if __name__ == "__main__":
    assembler = AssemblerPass1()
    try:
        pass1_results = assembler.run("functions.txt")
        print_pass1_results(pass1_results)
    except FileNotFoundError:
        print("Error: functions.txt must be in the same folder as this program.")
